# Rosie 兩句指定ID修正｜不重跑其他51段

若目前Colab執行階段仍在，**安裝格可跳過，從「修正準備」開始**。新執行階段才需先安裝。修正區塊不呼叫原先53段生成函式。

1. 修正準備：沿用原WORK參考包，若不存在才要求上傳原bakery-colab-inputs.zip。
2. 原音準備：沿用原53段輸出；若不存在才要求上傳你下載保存的53段輸出ZIP。
3. 候選生成：只生成指定兩句，每句3個候選；再次執行會建立新的獨立批次。
4. 比較與選擇：先聽主參考、原音、三候選，再用下拉選單選兩句，按匯出。沒有預選，不會自動套用。

如果想在目前筆記本中工作，可將以下修正區塊複製到原筆記本末尾；不要重新執行原本完整Bakery生成格。候選不保證5～6歲聽感，仍以你的驗收為準。Base使用聲線複製與隨機抽樣，沒有新增VoiceDesign、改英文、pitch或速度處理。

## 0｜僅新執行階段需要安裝；已成功安裝者跳過

In [ ]:
import sys, subprocess, os
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN']='1'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
subprocess.check_call(['apt-get','-qq','update'])
subprocess.check_call(['apt-get','-qq','install','-y','espeak-ng','sox','libsndfile1'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','qwen-tts==0.1.1','transformers==4.57.3','accelerate==1.12.0','kokoro==0.9.4','soundfile','huggingface-hub>=0.34,<1'])
print('套件安裝完成。若Colab提示套件已載入，請重新啟動執行階段，再從上傳格繼續。')

## 1｜修正準備（原輸入ZIP仍可使用）

In [ ]:
from pathlib import Path
from google.colab import files
import importlib.util, sys, json, uuid, shutil
MODULE_SOURCE="\"\"\"Targeted Rosie retakes. Originals are read-only; candidates and export are new folders.\"\"\"\nfrom pathlib import Path\nimport copy, datetime, hashlib, json, random, shutil, uuid, zipfile\n\nTARGETS = {\n    'hw.bakery.find-cookie.hint.01': 'Pumpkin. Find the pumpkin cookie.',\n    'hw.bakery.find-box.retry.01.n02': 'Good try. One pumpkin cookie and two candies.',\n}\nVOICE = 'qwen_rosie_r2'\nREF_HASH = 'f26a8e4ca3d1226a250e2507027f326f65060bd5cdf4d846e69c32913b051e4f'\nREF_TEXT = 'Welcome to the Halloween Bakery!'\n\ndef read(p):\n    return json.loads(Path(p).read_text(encoding='utf-8-sig'))\n\ndef sha(p):\n    return hashlib.sha256(Path(p).read_bytes()).hexdigest()\n\ndef write(p, data):\n    Path(p).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')\n\ndef inside(root, relative):\n    root = Path(root).resolve()\n    result = (root / relative).resolve()\n    if not result.is_relative_to(root):\n        raise ValueError('Path outside package')\n    return result\n\ndef unpack(raw, destination):\n    \"\"\"Extract uploaded zip into a NEW folder, never over existing recordings.\"\"\"\n    import io\n    destination = Path(destination)\n    if destination.exists():\n        raise ValueError('Upload destination already exists')\n    with zipfile.ZipFile(io.BytesIO(raw)) as archive:\n        if sum(i.file_size for i in archive.infolist()) > 300_000_000:\n            raise ValueError('ZIP exceeds 300 MB')\n        names = [i.filename for i in archive.infolist() if not i.is_dir()]\n        if len(names) != len(set(names)):\n            raise ValueError('Duplicate ZIP members')\n        paths = [(n, inside(destination, n)) for n in names]\n        destination.mkdir(parents=True)\n        for name, path in paths:\n            path.parent.mkdir(parents=True, exist_ok=True)\n            path.write_bytes(archive.read(name))\n    return destination\n\ndef plan(work, line_ids=None):\n    work = Path(work)\n    m = read(work / 'manifest.json')\n    c = m['characters']['rosie']\n    ref = c['reference']\n    if c['voiceId'] != VOICE or c['backend'] != 'qwen-base':\n        raise ValueError('Only the approved Rosie Qwen voice is allowed')\n    if ref['sha256'] != REF_HASH or ref['ref_text'] != REF_TEXT or ref['use'] != 'main':\n        raise ValueError('Approved Rosie reference/transcript mismatch')\n    if sha(inside(work / 'references', ref['file'])) != REF_HASH:\n        raise ValueError('Rosie reference audio changed')\n    ids = list(TARGETS if line_ids is None else line_ids)\n    if not ids or len(ids) != len(set(ids)) or any(i not in TARGETS for i in ids):\n        raise ValueError('This repair is limited to the two requested line IDs')\n    jobs = []\n    for line_id in ids:\n        line = m['lines'][line_id]\n        asset = m['assets'][line['assetId']]\n        if asset['role'] != 'rosie' or asset['voiceId'] != VOICE or asset['text'] != TARGETS[line_id]:\n            raise ValueError('Line identity or exact text mismatch')\n        jobs.append((line_id, asset))\n    return m, jobs\n\n_model = None\ndef load_model(lock):\n    global _model\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError('Select a free Colab T4 GPU; no paid fallback is used')\n    if _model is None:\n        from huggingface_hub import snapshot_download\n        from qwen_tts import Qwen3TTSModel\n        spec = lock['Qwen/Qwen3-TTS-12Hz-1.7B-Base']\n        path = snapshot_download(spec['repo'], revision=spec['revision'], token=False)\n        _model = Qwen3TTSModel.from_pretrained(path, device_map='cuda:0',\n            dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,\n            attn_implementation='sdpa')\n    return _model\n\ndef generate_candidates(work, parent, line_ids=None, count=3, run=False, model_factory=None):\n    m, jobs = plan(work, line_ids)\n    if count not in (2, 3):\n        raise ValueError('Choose 2 or 3 candidates per line')\n    if not run:\n        return [{'lineId':i, 'text':a['text'], 'voiceId':a['voiceId'], 'candidates':count} for i,a in jobs]\n    import numpy as np\n    import soundfile as sf\n    import torch\n    lock = read(Path(work) / 'model-lock.json')\n    model = (model_factory or load_model)(lock)\n    ref = m['characters']['rosie']['reference']\n    prompt = model.create_voice_clone_prompt(ref_audio=str(Path(work)/'references'/ref['file']),\n        ref_text=ref['ref_text'], x_vector_only_mode=False)\n    batch = Path(parent) / ('rosie-retakes-' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])\n    batch.mkdir(parents=True, exist_ok=False)\n    result = {'schemaVersion':1, 'scope':'bakery-rosie-retakes', 'source':m['source'],\n        'voiceId':VOICE, 'referenceSha256':REF_HASH, 'models':lock,\n        'productionApproved':False, 'candidates':{}}\n    write(batch/'candidate-manifest.json', result)\n    for line_id, asset in jobs:\n        result['candidates'][line_id] = []\n        for number in range(1, count+1):\n            seed = int.from_bytes(hashlib.sha256((batch.name+line_id+str(number)).encode()).digest()[:4], 'big')\n            random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n            if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n            settings = dict(do_sample=True, temperature=0.9, top_p=1.0, top_k=50,\n                subtalker_dosample=True, subtalker_temperature=0.9, max_new_tokens=2048)\n            wavs, sr = model.generate_voice_clone(text=asset['text'], language='English',\n                voice_clone_prompt=prompt, non_streaming_mode=True, **settings)\n            samples = np.asarray(wavs[0], dtype=np.float32)\n            if not np.isfinite(samples).all() or not 0.12 < len(samples)/sr < 45 or np.sqrt(np.mean(samples**2)) < .001:\n                raise ValueError('Invalid generated audio; completed candidates remain in '+str(batch))\n            gain = min(1., .98/max(float(np.max(np.abs(samples))), 1e-9))\n            name = f\"{asset['role']}__{line_id}__{VOICE}__candidate-{number}.wav\"\n            sf.write(batch/name, samples*gain, sr, subtype='PCM_16')\n            candidate = dict(assetId=asset['assetId'], lineId=line_id, text=asset['text'],\n                role='rosie', voiceId=VOICE, filename=name, sha256=sha(batch/name),\n                seed=seed, duration=len(samples)/sr, sampleRate=sr, gain=gain, sampling=settings,\n                referenceSha256=REF_HASH, review='pending-listener-choice', visemes=[])\n            result['candidates'][line_id].append(candidate)\n            write(batch/'candidate-manifest.json', result)\n            print(line_id, 'candidate', number, 'saved')\n    return batch\n\ndef validate_originals(work, original):\n    m, _ = plan(work)\n    original = Path(original)\n    old = read(original/'generated-manifest.json')\n    ids = set(read(Path(work)/'jobs.json')['bakery_complete'])\n    if old.get('scope') != 'bakery-generated-review' or old['source']['sha256'] != m['source']['sha256'] or set(old['assets']) != ids:\n        raise ValueError('Please provide the complete, matching 53-recording output')\n    for asset_id, a in old['assets'].items():\n        expected = m['assets'][asset_id]\n        if any(a.get(k) != expected[k] for k in ('role','voiceId','text','lineIds','src','filename')):\n            raise ValueError('Original asset identity mismatch: '+asset_id)\n        if sha(inside(original, a['src'])) != a['sha256']:\n            raise ValueError('Original WAV hash mismatch: '+asset_id)\n    return old\n\ndef export_selected(work, original, batch, choices, destination):\n    \"\"\"Write a full 53-file COPY. Change only selected assets in its manifest.\"\"\"\n    old = validate_originals(work, original)\n    m, _ = plan(work)\n    batch, destination = Path(batch), Path(destination)\n    if destination.exists() or destination.resolve().is_relative_to(Path(original).resolve()):\n        raise ValueError('Export must be a new independent directory')\n    c = read(batch/'candidate-manifest.json')\n    if c['source'] != m['source'] or c['voiceId'] != VOICE or c['referenceSha256'] != REF_HASH or c['models'] != read(Path(work)/'model-lock.json'):\n        raise ValueError('Candidate configuration mismatch')\n    if set(choices) != set(TARGETS):\n        raise ValueError('Choose one candidate for each requested line; no automatic choice')\n    replacements = {}\n    for line_id, number in choices.items():\n        if not isinstance(number, int) or isinstance(number, bool) or number < 1:\n            raise ValueError('Choose candidate 1, 2, or 3 after listening')\n        versions = c['candidates'].get(line_id, [])\n        if number > len(versions): raise ValueError('Candidate not generated')\n        a = versions[number-1]\n        expected = m['assets'][m['lines'][line_id]['assetId']]\n        if any(a.get(k) != expected[k] for k in ('assetId','role','voiceId','text')) or a['lineId'] != line_id or a['referenceSha256'] != REF_HASH:\n            raise ValueError('Candidate identity mismatch')\n        if sha(inside(batch, a['filename'])) != a['sha256']:\n            raise ValueError('Candidate WAV changed')\n        replacements[a['assetId']] = a\n    # Validate everything before writing any export. Original files never receive writes.\n    new = copy.deepcopy(old)\n    destination.mkdir(parents=True, exist_ok=False)\n    for asset_id, asset in old['assets'].items():\n        output = inside(destination, asset['src']); output.parent.mkdir(parents=True, exist_ok=True)\n        chosen = replacements.get(asset_id)\n        source = inside(batch, chosen['filename']) if chosen else inside(original, asset['src'])\n        shutil.copyfile(source, output)\n        if chosen:\n            new['assets'][asset_id].update({k:chosen[k] for k in ('sha256','duration','sampleRate','seed','gain','referenceSha256')})\n            new['assets'][asset_id].update(generationBackend='qwen-base',\n                generatedAt=datetime.datetime.now(datetime.timezone.utc).isoformat(),\n                review='user-selected-retake; in-game-review-pending', visemes=[],\n                retake={'lineId':chosen['lineId'], 'candidate':choices[chosen['lineId']],\n                        'previousSha256':asset['sha256'], 'sampling':chosen['sampling']})\n        if sha(output) != (chosen or asset)['sha256']: raise ValueError('Copy hash mismatch')\n    write(destination/'generated-manifest.json', new)\n    write(destination/'repair-selection.json', {'choices':choices, 'changedAssetIds':list(replacements),\n        'unchangedAssets':51, 'originalManifestSha256':sha(Path(original)/'generated-manifest.json'),\n        'productionApproved':False})\n    return destination\n"
module_path=Path('/content/rosie_repair_runner.py')
module_path.write_text(MODULE_SOURCE,encoding='utf-8')
spec=importlib.util.spec_from_file_location('rosie_repair_runner',module_path)
repair=importlib.util.module_from_spec(spec);spec.loader.exec_module(repair)
WORK=Path(globals().get('WORK','/content/rosie_bakery_work'))
if not (WORK/'manifest.json').exists():
    print('請上傳原 bakery-colab-inputs.zip（私人參考包）')
    uploaded=files.upload()
    if len(uploaded)!=1: raise ValueError('請只上傳一個輸入ZIP')
    WORK=repair.unpack(next(iter(uploaded.values())),Path('/content')/('bakery_repair_inputs_'+uuid.uuid4().hex[:8]))
manifest,jobs=repair.plan(WORK)
for item in repair.generate_candidates(WORK,'/content/rosie_bakery_repairs',run=False):print(item)
print('核准Rosie參考音、逐字稿及兩個固定ID已核對。尚未生成。')

## 2｜載入原本完整53段（唯讀，不會覆蓋）

In [ ]:
ORIGINAL=Path(globals().get('OUTPUT','/content/rosie_bakery_output'))
if not (ORIGINAL/'generated-manifest.json').exists():
    print('請上傳先前下載的完整53段輸出ZIP，並非輸入參考包')
    uploaded=files.upload()
    if len(uploaded)!=1:raise ValueError('請只上傳一個53段輸出ZIP')
    ORIGINAL=repair.unpack(next(iter(uploaded.values())),Path('/content')/('bakery_original_53_'+uuid.uuid4().hex[:8]))
original_manifest=repair.validate_originals(WORK,ORIGINAL)
print('完整53段與雜湊已驗證；來源資料夾保持唯讀：',ORIGINAL)

## 3｜只生成兩句，各三候選（共6段）

In [ ]:
# 執行此格才會載入模型並推論。不會呼叫原53段批次生成。
TARGET_LINE_IDS=list(repair.TARGETS)
CANDIDATES_PER_LINE=3
BATCH=repair.generate_candidates(WORK,Path('/content/rosie_bakery_repairs'),
    line_ids=TARGET_LINE_IDS,count=CANDIDATES_PER_LINE,run=True)
print('獨立候選資料夾：',BATCH)
print('原53段、原manifest均未修改。')

## 4｜比較原音與候選，選定後才匯出新53段副本

In [ ]:
from IPython.display import Audio, display
import ipywidgets as widgets
review=repair.read(BATCH/'candidate-manifest.json')
ref=manifest['characters']['rosie']['reference']
print('先聽核准Rosie主參考：',ref['ref_text'])
display(Audio(filename=str(WORK/'references'/ref['file'])))
selectors={}
for line_id in TARGET_LINE_IDS:
    a=manifest['assets'][manifest['lines'][line_id]['assetId']]
    print('\n',line_id,'\n',a['text'],'\n原音：')
    display(Audio(filename=str(ORIGINAL/a['src'])))
    versions=review['candidates'][line_id]
    for n,c in enumerate(versions,1):
        print('候選',n,'／seed',c['seed'])
        display(Audio(filename=str(BATCH/c['filename'])))
    selectors[line_id]=widgets.Dropdown(options=[('尚未選擇',None)]+[('候選 '+str(n),n) for n in range(1,len(versions)+1)],description='採用版本：')
    display(selectors[line_id])
status=widgets.Output()
export_button=widgets.Button(description='確認選擇並匯出修正副本',layout=widgets.Layout(width='240px'))
download_button=widgets.Button(description='只下載候選、不採用',layout=widgets.Layout(width='240px'))
def download_candidates(_):
    with status:
        files.download(shutil.make_archive(str(BATCH),'zip',BATCH))
def export_choices(_):
    with status:
        status.clear_output()
        choices={i:selector.value for i,selector in selectors.items()}
        if any(v is None for v in choices.values()):
            print('請先為兩句各選一個喜歡的版本；若都不喜歡可只下載候選或重跑區塊3。');return
        try:
            target=Path('/content/rosie_bakery_repairs')/('selected_53_'+uuid.uuid4().hex[:8])
            repair.export_selected(WORK,ORIGINAL,BATCH,choices,target)
            print('新副本：',target,'；51段逐檔不變，2段採用選定候選。原資料未覆蓋。')
            files.download(shutil.make_archive(str(target),'zip',target))
        except Exception as exc:print('未完成匯出：',exc)
export_button.on_click(export_choices);download_button.on_click(download_candidates)
display(download_button,export_button,status)

## 驗證與限制

修正候選manifest保留原lineId/assetId/英文/聲線/參考雜湊；選定後僅在新資料夾的generated-manifest.json更新兩個音檔紀錄。原51段保持音檔與manifest紀錄相同。選定候選清除過時visemes，仍須遊戲內同步驗收，不是精準音素同步。

已做本機模擬測試與Notebook語法檢查；本次未實際執行Colab GPU、未產生或試聽6段新候選。Qwen介面依[官方文件](https://github.com/QwenLM/Qwen3-TTS)。免費GPU可用性受Colab限制。輸入參考音不會公開，正式遊戲不修改。